# `mart_county_features` — Data Dictionary

The `main_marts.mart_county_features` table is the wide-form output of the emerging-RE feature pipeline.

**Grain:** one row per `(fips, period_end_date)` — where `period_end_date` is Dec-31 of each cohort year.

**Row count:** ~3,200 counties × 15+ cohort years ≈ **48,000+ rows**.

**Columns:** 100 feature columns + 6 targets (grouped below by theme).

**Point-in-time (PIT) correctness:** As of 2026-09-02, ACS-derived features (rental vacancy, rent-to-income, cap rate proxy, renter share, MF permit ratio) are rebuilt PIT-safe. ⚠️ noted below are still latest-per-fips and should be treated as leaky for ML training.

**Auto-generated stats** in section 2 below (null rate, min/max, sample values).

## 0. Setup

In [ ]:
import duckdb, pandas as pd
from pathlib import Path

WAREHOUSE = "/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb"
con = duckdb.connect(WAREHOUSE, read_only=True)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)

## 1. Curated data dictionary — grouped by theme

Every column with:
- **Source model** — which `int_*` model produces it
- **Description** — semantic meaning
- **Unit** — share / count / $ / bool / pp / etc.

In [ ]:
DICTIONARY = [
    ('=== IDENTITY ===', None, None, None),
    ('fips', 'int_universe', '5-digit county FIPS code (state+county)', 'text'),
    ('period_end_date', 'int_universe', 'Point-in-time scoring date (always Dec-31 for annual cohorts)', 'date'),
    ('county_name', 'int_universe', 'County name (Census Gazetteer)', 'text'),
    ('state_name', 'int_universe', 'State name (Census Gazetteer)', 'text'),
    ('pop_latest', 'int_census_pop_estimates', 'Latest annual population estimate (Census PEP) as of scoring date', 'count'),
    ('=== MIGRATION ===', None, None, None),
    ('domesticmig_per_1000', 'int_census_pop_estimates', 'Net domestic in-migration per 1,000 residents (annual)', 'per 1,000'),
    ('domesticmig_accel_3yr', 'int_census_pop_estimates', '3-year change in domestic migration rate — is inflow accelerating?', 'per 1,000 Δ'),
    ('internationalmig', 'int_census_pop_estimates', 'Absolute annual international net migration', 'count'),
    ('internationalmig_per_1000', 'int_census_pop_estimates', 'International net migration per 1,000 residents', 'per 1,000'),
    ('irs_net_migration_rate', 'int_irs_migration', 'IRS SOI: net taxpayer migration rate (returns_in − returns_out) / total', 'share'),
    ('irs_inflow_rate', 'int_irs_migration', 'IRS SOI: gross inflow rate of taxpayer returns', 'share'),
    ('irs_income_gap', 'int_irs_migration', 'AGI gap: avg AGI of inflow returns − avg AGI of outflow returns ($)', '$'),
    ('irs_income_gap_pct', 'int_irs_migration', 'Same gap as % of outflow AGI baseline', '%'),
    ('irs_net_agi_per_capita', 'int_irs_migration', 'Net AGI flow per capita — dollar impact of migration', '$/person'),
    ('=== INCOME & WAGES ===', None, None, None),
    ('per_capita_income', 'int_bea_income', 'BEA CAINC1 per-capita income at scoring date (nominal $)', '$'),
    ('pcpi_yoy', 'int_bea_income', 'PCPI YoY change', 'share'),
    ('pcpi_cagr_3yr', 'int_bea_income', '3-year CAGR of per-capita income', 'share/yr'),
    ('pcpi_cagr_5yr', 'int_bea_income', '5-year CAGR of per-capita income (⚠️ may be latest-per-fips, treat as leaky for training)', 'share/yr'),
    ('pcpi_growth_vs_state_1yr_pp', 'int_bea_income', '1yr PCPI growth minus state-mean growth (percentage points)', 'pp'),
    ('pcpi_cagr_3yr_vs_state_pp', 'int_bea_income', '3yr PCPI CAGR minus state-mean CAGR', 'pp'),
    ('wage_growth_yoy', 'int_qcew_growth', 'QCEW avg weekly wage YoY', 'share'),
    ('wage_growth_yoy_demeaned', 'int_qcew_growth', 'Wage growth minus national mean (level shift)', 'share'),
    ('high_wage_share', 'int_qcew_growth', 'Employment share in high-wage sectors (info, finance, prof/tech, mgmt)', 'share'),
    ('high_wage_share_yoy_chg', 'int_qcew_growth', 'YoY change in high-wage employment share', 'share Δ'),
    ('=== EMPLOYMENT ===', None, None, None),
    ('emp_growth_yoy', 'int_qcew_growth', 'QCEW total employment YoY', 'share'),
    ('emp_growth_yoy_demeaned', 'int_qcew_growth', 'Employment growth minus national mean', 'share'),
    ('unemployment_rate', 'int_bls_laus_unemployment', 'BLS LAUS unemployment rate (annual avg)', 'share'),
    ('unemployment_rate_yoy_chg', 'int_bls_laus_unemployment', 'YoY change in unemployment rate', 'share Δ'),
    ('knowledge_emp_yoy', 'int_qcew_knowledge', 'YoY growth in knowledge-sector employment (NAICS 51,54,55)', 'share'),
    ('knowledge_emp_cagr_3yr', 'int_qcew_knowledge', '3yr CAGR of knowledge-sector employment', 'share/yr'),
    ('knowledge_emp_growth_vs_state_pp', 'int_qcew_knowledge', 'Knowledge emp growth minus state mean', 'pp'),
    ('renter_ind_emp_yoy', 'int_mf_features', 'YoY growth in renter-heavy industry employment (retail, food, healthcare, admin support)', 'share'),
    ('renter_ind_emp_cagr_3yr', 'int_mf_features', '3yr CAGR renter-industry employment', 'share/yr'),
    ('renter_ind_emp_growth_vs_state_pp', 'int_mf_features', 'Same, minus state mean', 'pp'),
    ('=== EMPLOYMENT DIVERSITY / ANCHORS ===', None, None, None),
    ('employment_diversity_hhi', 'int_employment_diversity', 'Herfindahl-Hirschman Index of QCEW supersector employment shares. 0 = perfectly diverse, 1 = single-sector monopoly. <0.10 = well-diversified', 'index'),
    ('n_supersectors_reporting', 'int_employment_diversity', 'Count of QCEW supersectors with non-null employment (data-quality indicator; ⚠️ may be latest-per-fips)', 'count'),
    ('max_supersector_share', 'int_employment_diversity', "Largest single supersector's share of employment", 'share'),
    ('anchor_type', 'int_employment_diversity', "Categorical: 'gov'/'edu'/'healthcare'/'tourism'/'none' — dominant institutional anchor", 'categorical'),
    ('anchor_stability_score', 'int_employment_diversity', 'Score 0-1 rating anchor stability (higher = more resilient)', 'score'),
    ('federal_gov_share', 'int_employment_diversity', 'Federal government employment share (military bases, VA hospitals, etc.)', 'share'),
    ('edu_share', 'int_employment_diversity', 'Education services employment share (NAICS 611)', 'share'),
    ('healthcare_share', 'int_employment_diversity', 'Healthcare & social assistance employment share (NAICS 62)', 'share'),
    ('tourism_share', 'int_employment_diversity', 'Accommodation + food services employment share (NAICS 72)', 'share'),
    ('=== DEMOGRAPHICS ===', None, None, None),
    ('age_20_34_share', 'int_acs_demographics', 'Share of population aged 20-34 (renter-heavy cohort) from ACS', 'share'),
    ('age_20_34_share_yoy_chg', 'int_acs_demographics', 'YoY change in 20-34 share', 'share Δ'),
    ('age_20_34_growth_3yr_cagr', 'int_acs_demographics', '3yr CAGR of 20-34 population share', 'share/yr'),
    ('foreign_born_share', 'int_acs_demographics', 'Foreign-born population share (ACS)', 'share'),
    ('foreign_born_share_yoy_chg', 'int_acs_demographics', 'YoY change', 'share Δ'),
    ('foreign_born_cagr_3yr', 'int_acs_demographics', '3yr CAGR of foreign-born share', 'share/yr'),
    ('=== HOUSING SUPPLY & PERMITS ===', None, None, None),
    ('permits_per_1000', 'int_bps_permits', 'Building permits (total units) per 1,000 residents, annual', 'per 1,000'),
    ('permits_per_1000_10yr_baseline', 'int_bps_permits', '10-year trailing mean of permits_per_1000 (baseline)', 'per 1,000'),
    ('permits_intensity_ratio', 'int_bps_permits', 'Current permits_per_1000 / 10yr baseline (>1 = elevated supply)', 'ratio'),
    ('permits_10yr_baseline_years_available', 'int_bps_permits', 'Number of years in the trailing baseline window (data quality)', 'count'),
    ('mf_permit_ratio', 'int_mf_features', "5+ unit permits / total permits (share of new supply that's MF)", 'share'),
    ('mf_permit_ratio_ts', 'int_mf_features', 'Time-series version of mf_permit_ratio (rolling)', 'share'),
    ('mf_permit_ratio_change_3yr', 'int_mf_features', '3yr change in mf_permit_ratio', 'share Δ'),
    ('enp_yoy', 'int_enp_permits', 'Emerging non-permitted (ENP) growth YoY — proxy for informal MF development', 'share'),
    ('enp_cagr_3yr', 'int_enp_permits', '3yr CAGR of ENP', 'share/yr'),
    ('enp_growth_vs_natl_pp', 'int_enp_permits', 'ENP growth minus national mean', 'pp'),
    ('business_applications', 'int_bfs_business_apps', 'Census BFS business applications count (annual)', 'count'),
    ('business_apps_per_1000', 'int_bfs_business_apps', 'Business applications per 1,000 residents', 'per 1,000'),
    ('business_apps_per_1000_3yr_avg', 'int_bfs_business_apps', '3yr rolling avg of business apps per 1,000', 'per 1,000'),
    ('=== HOME PRICES & RENTS ===', None, None, None),
    ('hpi_yoy', 'int_hpi_features', 'FHFA HPI YoY change (all-transactions)', 'share'),
    ('zori_yoy', 'int_zori_features', 'Zillow Observed Rent Index YoY change', 'share'),
    ('hpi_trailing_24mo', 'int_hpi_features', 'HPI cumulative return over trailing 24 months', 'share'),
    ('hpi_5yr_cagr_deficit', 'int_hpi_features', 'Current 5yr HPI CAGR minus long-run avg — negative = underperformance', 'pp'),
    ('zori_5yr_cagr_deficit', 'int_zori_features', 'Current 5yr ZORI CAGR minus long-run avg', 'pp'),
    ('rent_minus_price_growth', 'mart_county_features', 'zori_yoy − hpi_yoy (positive = rents growing faster than prices → cap expansion)', 'pp'),
    ('rent_growth_vs_wage_growth', 'mart_county_features', 'zori_yoy − wage_growth_yoy (positive = rent burden increasing)', 'pp'),
    ('=== MF & RENTAL MARKET ===', None, None, None),
    ('renter_share', 'int_mf_features', 'Share of occupied housing units that are renter-occupied (ACS)', 'share'),
    ('rental_vacancy_rate', 'int_acs_features', 'Rental vacancy rate (ACS)', 'share'),
    ('rental_vacancy_change_3yr', 'int_mf_features', '3yr change in rental vacancy rate (negative = tightening)', 'share Δ'),
    ('rent_to_income_ratio', 'int_acs_features', 'Median gross rent × 12 / median household income', 'ratio'),
    ('cap_rate_proxy', 'int_acs_features', 'ACS proxy cap rate: median_gross_rent × 12 / median_home_value (SFR+MF blended, includes utilities)', 'share'),
    ('cap_rate_now_ts', 'int_mf_features', 'Time-series version of cap rate proxy', 'share'),
    ('mf_cap_rate_estimated', 'int_mf_features', 'MF-adjusted cap: (HUD FMR 2BR × 12 / median_home_value) + 1.5% MF premium — closer to acquisition caps', 'share'),
    ('mf_cap_rate_change_3yr', 'int_mf_features', '3yr change in mf_cap_rate_estimated', 'share Δ'),
    ('fmr_2br', 'int_hud_fmr', 'HUD Fair Market Rent, 2-bedroom (40th percentile gross rent)', '$/mo'),
    ('fmr_2br_yoy', 'int_hud_fmr', 'FMR 2BR YoY change', 'share'),
    ('fmr_2br_cagr_3yr', 'int_hud_fmr', 'FMR 2BR 3yr CAGR', 'share/yr'),
    ('fmr_2br_cagr_3yr_vs_state_pp', 'int_hud_fmr', 'FMR CAGR minus state-mean CAGR', 'pp'),
    ('vt_vacancy_level', 'int_zip_vacancy', 'USPS ZIP-level vacancy rate (Valassis) — often null (HUD blocked)', 'share'),
    ('vt_vacancy_change_3yr', 'int_zip_vacancy', '3yr change in vacancy level', 'share Δ'),
    ('vt_vacancy_change_5yr', 'int_zip_vacancy', '5yr change in vacancy level', 'share Δ'),
    ('=== DISQUALIFIERS (boolean gates) ===', None, None, None),
    ('disq_aging_trap', 'mart_county_features', 'TRUE if 20-34 share is declining AND foreign_born share is declining (demographic decay)', 'bool'),
    ('disq_sector_concentration', 'mart_county_features', 'TRUE if max_supersector_share > threshold (single-industry town)', 'bool'),
    ('disq_price_ran_25pct', 'mart_county_features', 'TRUE if HPI already ran +25% in trailing 24mo (top-tick risk)', 'bool'),
    ('any_disqualifier', 'mart_county_features', 'TRUE if ANY disq_* flag is set — top-level filter', 'bool'),
    ('=== FORWARD RETURNS & ML TARGETS ===', None, None, None),
    ('hpi_forward_return_24mo', 'mart_county_features', 'Realized 24mo HPI return from period_end_date (LEAKS if used as feature)', 'share'),
    ('hpi_forward_return_36mo', 'mart_county_features', 'Realized 36mo HPI return', 'share'),
    ('hpi_forward_return_60mo', 'mart_county_features', 'Realized 60mo HPI return', 'share'),
    ('zori_forward_return_24mo', 'mart_county_features', 'Realized 24mo ZORI rent return', 'share'),
    ('zori_forward_return_36mo', 'mart_county_features', 'Realized 36mo ZORI rent return', 'share'),
    ('zori_forward_return_60mo', 'mart_county_features', 'Realized 60mo ZORI rent return', 'share'),
    ('target_hpi_24mo_demeaned', 'mart_county_features', 'hpi_forward_return_24mo minus cohort-year national mean — ML training target', 'share'),
    ('target_hpi_36mo_demeaned', 'mart_county_features', '36mo demeaned HPI target', 'share'),
    ('target_hpi_60mo_demeaned', 'mart_county_features', '60mo demeaned HPI target', 'share'),
    ('target_zori_24mo_demeaned', 'mart_county_features', '24mo demeaned ZORI target', 'share'),
    ('target_zori_36mo_demeaned', 'mart_county_features', '36mo demeaned ZORI target — primary ML target used by analog_search', 'share'),
    ('target_zori_60mo_demeaned', 'mart_county_features', '60mo demeaned ZORI target', 'share'),
]

rows = []
current_section = None
for col, src, desc, unit in DICTIONARY:
    if col.startswith('==='):
        current_section = col.strip('= ').strip()
        continue
    rows.append({
        'section': current_section, 'column': col, 'source_model': src,
        'unit': unit, 'description': desc,
    })
dd = pd.DataFrame(rows)
print(f"Total curated columns: {len(dd)}")
dd

## 2. Auto-generated stats — verify coverage per column

For each column: dtype, non-null count, null%, min/max/mean (numeric), sample values (categorical).

In [ ]:
# Column types + non-null counts
schema = con.execute('DESCRIBE main_marts.mart_county_features').df()[['column_name','column_type']]

# Row count + null counts per column
n_rows = con.execute('SELECT COUNT(*) FROM main_marts.mart_county_features').fetchone()[0]

null_query = "SELECT " + ", ".join(f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END) AS "{c}"' for c in schema['column_name']) + " FROM main_marts.mart_county_features"
nulls = con.execute(null_query).df().T.reset_index()
nulls.columns = ['column_name','n_null']

stats = schema.merge(nulls, on='column_name')
stats['n_non_null'] = n_rows - stats['n_null']
stats['null_pct'] = (stats['n_null'] / n_rows * 100).round(1)
print(f"Total rows in mart: {n_rows:,}")
stats[['column_name','column_type','n_non_null','null_pct']]

In [ ]:
# Numeric distributions — min/median/max/mean for numeric columns
numeric_cols = schema[schema['column_type'].isin(['DOUBLE','BIGINT','INTEGER'])]['column_name'].tolist()
parts = []
for c in numeric_cols:
    parts.append(f'MIN("{c}") AS "{c}__min", '
                 f'PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY "{c}") AS "{c}__p50", '
                 f'MAX("{c}") AS "{c}__max", '
                 f'AVG("{c}") AS "{c}__mean"')
q = "SELECT " + ", ".join(parts) + " FROM main_marts.mart_county_features"
r = con.execute(q).df()

# Reshape to one row per column
records = []
for c in numeric_cols:
    records.append({'column': c, 'min': r[f'{c}__min'][0], 'p50': r[f'{c}__p50'][0],
                    'max': r[f'{c}__max'][0], 'mean': r[f'{c}__mean'][0]})
num_dist = pd.DataFrame(records)
num_dist

In [ ]:
# Categorical sample values — for VARCHAR + BOOLEAN columns
cat_cols = schema[schema['column_type'].isin(['VARCHAR','BOOLEAN'])]['column_name'].tolist()
cat_samples = []
for c in cat_cols:
    if c in ('fips','county_name','state_name','period_end_date'):
        continue  # skip identity columns
    top = con.execute(f'SELECT "{c}", COUNT(*) AS n FROM main_marts.mart_county_features GROUP BY 1 ORDER BY n DESC LIMIT 5').df()
    cat_samples.append({'column': c, 'top_5_values': top.to_dict(orient='records')})
pd.DataFrame(cat_samples)

## 3. Merged view — dictionary + stats + coverage side-by-side

In [ ]:
merged = dd.merge(stats[['column_name','column_type','null_pct']],
                  left_on='column', right_on='column_name', how='left').drop(columns=['column_name'])
merged = merged[['section','column','column_type','null_pct','unit','source_model','description']]
merged

## 4. Sources — where the data actually comes from

Every mart column ultimately traces back to one of these raw sources. Below: one entry per source with the exact fields extracted and the mart features they feed.

Pipeline: `ingest/*.py` → `stg_*` (DuckDB staging) → `int_*` (dbt intermediates) → `mart_county_features`.

In [ ]:
source_summary = [
    {'source': 'ACS 5-Year Estimates', 'agency': 'U.S. Census Bureau', 'staging_table': 'stg_census_acs', 'grain': 'county × 5yr-vintage (bitemporal)', 'coverage': '~48K rows, ~3,200 counties × ~15 vintages (2005-2023)', 'n_fields_extracted': 8, 'intermediate_models': 'int_acs_features', 'n_mart_features': 4, 'notes': '5-year rolling aggregate — smoother but lagged. All counties. PIT-aware after 2026-09-02 rewrite.'},
    {'source': 'ACS 1-Year Estimates', 'agency': 'U.S. Census Bureau', 'staging_table': 'stg_census_acs_1yr', 'grain': 'county × annual (pop ≥ 65K only)', 'coverage': '~7.5K rows, ~1,000 large counties × 2010-2024', 'n_fields_extracted': 9, 'intermediate_models': 'int_acs_demographics + int_mf_inventory + int_mf_features', 'n_mart_features': 5, 'notes': 'Fresher than 5yr but only pop ≥ 65K. Only source of MF stock breakdown by building size.'},
    {'source': 'Population Estimates Program (PEP)', 'agency': 'U.S. Census Bureau', 'staging_table': 'stg_census_pep', 'grain': 'county × annual', 'coverage': '~50K rows, ~3,200 counties × 2000-2024', 'n_fields_extracted': 5, 'intermediate_models': 'int_census_pop_estimates', 'n_mart_features': 3, 'notes': 'Primary pop + migration source. Used for per-capita normalizations across the mart.'},
    {'source': 'Building Permits Survey (BPS)', 'agency': 'U.S. Census Bureau', 'staging_table': 'stg_census_bps', 'grain': 'county × annual', 'coverage': '~33K rows. ⚠️ Coverage cliff: ~750 counties pre-2022, ~3,000 from 2022 onward', 'n_fields_extracted': 4, 'intermediate_models': 'int_bps_permits + int_mf_features', 'n_mart_features': 4, 'notes': 'Only source of new-supply pipeline. Pre-2022 coverage cliff limits historical backtest.'},
    {'source': 'Business Formation Statistics (BFS)', 'agency': 'U.S. Census Bureau', 'staging_table': 'stg_census_bfs', 'grain': 'county × annual', 'coverage': '~66K rows', 'n_fields_extracted': 1, 'intermediate_models': 'int_bfs_business_apps', 'n_mart_features': 3, 'notes': 'Proxy for entrepreneurial energy / small-business formation.'},
    {'source': 'Quarterly Census of Employment and Wages (QCEW)', 'agency': 'U.S. Bureau of Labor Statistics', 'staging_table': 'stg_bls_qcew', 'grain': 'county × NAICS supersector × annual', 'coverage': '~2M rows — deepest employment/wage source', 'n_fields_extracted': 6, 'intermediate_models': 'int_qcew_growth + int_qcew_knowledge + int_employment_diversity + int_mf_features (renter-industry emp)', 'n_mart_features': 9, 'notes': 'Backbone of employment/wage/diversity features. Covers UI-covered employees (very complete).'},
    {'source': 'Local Area Unemployment Statistics (LAUS)', 'agency': 'U.S. Bureau of Labor Statistics', 'staging_table': 'stg_bls_laus', 'grain': 'county × month (annualized)', 'coverage': '~1.4M rows', 'n_fields_extracted': 1, 'intermediate_models': 'int_bls_laus_unemployment', 'n_mart_features': 2, 'notes': 'Monthly county unemployment, aggregated to annual for mart.'},
    {'source': 'Regional Income (CAINC1)', 'agency': 'U.S. Bureau of Economic Analysis', 'staging_table': 'stg_bea_regional_income', 'grain': 'county × annual', 'coverage': '~19K rows', 'n_fields_extracted': 3, 'intermediate_models': 'int_bea_income', 'n_mart_features': 3, 'notes': 'Primary per-capita income levels & growth. Own pop denominator (differs slightly from PEP).'},
    {'source': 'Fair Market Rents (FMR)', 'agency': 'U.S. Department of Housing and Urban Development', 'staging_table': 'stg_hud_fmr', 'grain': 'county × annual (fiscal year)', 'coverage': '~23K rows', 'n_fields_extracted': 2, 'intermediate_models': 'int_hud_fmr + int_mf_features (mf_cap_rate_estimated)', 'n_mart_features': 4, 'notes': 'HUD 40th-%ile 2BR gross rent. Slow annual update. Backbone of MF-adjusted cap proxy.'},
    {'source': 'House Price Index (HPI)', 'agency': 'Federal Housing Finance Agency (FHFA)', 'staging_table': 'stg_fhfa_hpi', 'grain': 'county × quarterly (annualized)', 'coverage': '~105K rows', 'n_fields_extracted': 2, 'intermediate_models': 'int_hpi_features', 'n_mart_features': 6, 'notes': 'Only source of long-term home price series. Repeat-sales, SFR-heavy but best available.'},
    {'source': 'Zillow Observed Rent Index (ZORI)', 'agency': 'Zillow (private)', 'staging_table': 'stg_zillow_zori', 'grain': 'county × monthly (annualized)', 'coverage': '~93K rows. Coverage improves post-2015; sparse pre-2015', 'n_fields_extracted': 1, 'intermediate_models': 'int_zori_features', 'n_mart_features': 4, 'notes': 'Monthly, MF-adjacent. Complements ACS/FMR by being fresher but less deep.'},
    {'source': 'IRS SOI Migration', 'agency': 'U.S. Internal Revenue Service', 'staging_table': 'stg_irs_soi_migration', 'grain': 'county × annual', 'coverage': '~12K rows', 'n_fields_extracted': 3, 'intermediate_models': 'int_irs_migration', 'n_mart_features': 5, 'notes': 'Taxpayer-return migration flows with AGI — reveals income quality of movers.'},
    {'source': 'National Risk Index (NRI)', 'agency': 'FEMA', 'staging_table': 'stg_fema_nri', 'grain': 'county (snapshot)', 'coverage': '~3,200 counties × 1 vintage', 'n_fields_extracted': 4, 'intermediate_models': '(not yet integrated)', 'n_mart_features': 1, 'notes': 'Loaded but not yet integrated. Available for future climate-adjusted underwriting.'},
    {'source': 'FAA Enplanements', 'agency': 'Federal Aviation Administration', 'staging_table': 'stg_faa_enplanements', 'grain': 'county × annual', 'coverage': '~2.9K rows', 'n_fields_extracted': 3, 'intermediate_models': 'int_enp_permits', 'n_mart_features': 1, 'notes': 'Airport passenger traffic — proxy for connectivity/tourism/business travel. Only counties with commercial airports.'},
    {'source': 'FRED Macro Series', 'agency': 'Federal Reserve Bank of St. Louis', 'staging_table': 'stg_fred_macro', 'grain': 'national × series × observation_date', 'coverage': '~30K rows across multiple national series', 'n_fields_extracted': 1, 'intermediate_models': 'int_national_macro_regime', 'n_mart_features': 1, 'notes': 'National context. Tags each cohort year with a rate regime for tie-breaker analysis.'},
    {'source': 'HUD USPS Vacancy', 'agency': 'U.S. Department of Housing and Urban Development', 'staging_table': '(not loaded — blocked)', 'grain': 'ZIP × quarterly (would be)', 'coverage': 'None — requires HUD gov/nonprofit/edu-only USPS Vacancies registration', 'n_fields_extracted': 1, 'intermediate_models': 'int_zip_vacancy (skeleton exists)', 'n_mart_features': 1, 'notes': '⚠️ Deferred — HUD registration barrier. Framework ready; awaiting access.'},
]

import pandas as pd
src_df = pd.DataFrame(source_summary)
pd.set_option('display.max_colwidth', 80)
src_df

### 4.1 Detailed field-by-source breakdown

For each source: what raw fields we pull, and which mart columns they feed.

#### ACS 5-Year Estimates — *U.S. Census Bureau*

- **Staging table:** `stg_census_acs`
- **Grain:** county × 5yr-vintage (bitemporal)
- **Coverage:** ~48K rows, ~3,200 counties × ~15 vintages (2005-2023)
- **Intermediate models:** `int_acs_features`

**Fields extracted from source (8):**

- `median_household_income`
- `median_gross_rent`
- `median_home_value`
- `owner_occupied_units`
- `renter_occupied_units`
- `vacant_for_rent_units`
- `rental_vacancy_rate`
- `total_population`

**Mart columns fed by this source (4):**

- `rental_vacancy_rate`
- `rent_to_income_ratio`
- `cap_rate_proxy`
- `acs_forward_return_36mo (target)`

**Notes:** 5-year rolling aggregate — smoother but lagged. All counties. PIT-aware after 2026-09-02 rewrite.


#### ACS 1-Year Estimates — *U.S. Census Bureau*

- **Staging table:** `stg_census_acs_1yr`
- **Grain:** county × annual (pop ≥ 65K only)
- **Coverage:** ~7.5K rows, ~1,000 large counties × 2010-2024
- **Intermediate models:** `int_acs_demographics` · `int_mf_inventory` · `int_mf_features`

**Fields extracted from source (9):**

- `median_household_income`
- `median_gross_rent`
- `median_home_value`
- `rental_vacancy_rate`
- `total_population`
- `age_20_34`
- `foreign_born_population`
- `units_in_5_to_9_bldgs / 10_to_19 / 20_to_49 / 50_plus (B25024 MF stock)`
- `hu_built_2020_plus … hu_built_pre_1940 (B25034 housing age)`

**Mart columns fed by this source (5):**

- `age_20_34_share, age_20_34_share_yoy_chg, age_20_34_growth_3yr_cagr`
- `foreign_born_share, foreign_born_share_yoy_chg, foreign_born_cagr_3yr`
- `renter_share`
- `MF unit counts (5-9, 10-19, 20-49, 50+) and building estimates`
- `pre-1980/pre-1970/pre-1960 housing share`

**Notes:** Fresher than 5yr but only pop ≥ 65K. Only source of MF stock breakdown by building size.


#### Population Estimates Program (PEP) — *U.S. Census Bureau*

- **Staging table:** `stg_census_pep`
- **Grain:** county × annual
- **Coverage:** ~50K rows, ~3,200 counties × 2000-2024
- **Intermediate models:** `int_census_pop_estimates`

**Fields extracted from source (5):**

- `popestimate`
- `births, deaths, naturalchg`
- `internationalmig (net international)`
- `domesticmig (net domestic)`
- `netmig, npopchg`

**Mart columns fed by this source (3):**

- `pop_latest`
- `domesticmig_per_1000, domesticmig_accel_3yr`
- `internationalmig, internationalmig_per_1000`

**Notes:** Primary pop + migration source. Used for per-capita normalizations across the mart.


#### Building Permits Survey (BPS) — *U.S. Census Bureau*

- **Staging table:** `stg_census_bps`
- **Grain:** county × annual
- **Coverage:** ~33K rows. ⚠️ Coverage cliff: ~750 counties pre-2022, ~3,000 from 2022 onward
- **Intermediate models:** `int_bps_permits` · `int_mf_features`

**Fields extracted from source (4):**

- `units_1 (SFR)`
- `units_2, units_34 (small MF)`
- `units_5plus (MF permits — target segment)`
- `units_total, bldgs_total, value_total`

**Mart columns fed by this source (4):**

- `permits_per_1000, permits_per_1000_10yr_baseline`
- `permits_intensity_ratio`
- `mf_permit_ratio (5+ units / total)`
- `mf_permit_ratio_ts, mf_permit_ratio_change_3yr`

**Notes:** Only source of new-supply pipeline. Pre-2022 coverage cliff limits historical backtest.


#### Business Formation Statistics (BFS) — *U.S. Census Bureau*

- **Staging table:** `stg_census_bfs`
- **Grain:** county × annual
- **Coverage:** ~66K rows
- **Intermediate models:** `int_bfs_business_apps`

**Fields extracted from source (1):**

- `business_applications (annual count)`

**Mart columns fed by this source (3):**

- `business_applications`
- `business_apps_per_1000`
- `business_apps_per_1000_3yr_avg`

**Notes:** Proxy for entrepreneurial energy / small-business formation.


#### Quarterly Census of Employment and Wages (QCEW) — *U.S. Bureau of Labor Statistics*

- **Staging table:** `stg_bls_qcew`
- **Grain:** county × NAICS supersector × annual
- **Coverage:** ~2M rows — deepest employment/wage source
- **Intermediate models:** `int_qcew_growth` · `int_qcew_knowledge` · `int_employment_diversity` · `int_mf_features (renter-industry emp)`

**Fields extracted from source (6):**

- `annual_avg_estabs`
- `annual_avg_emplvl`
- `total_annual_wages`
- `annual_avg_wkly_wage`
- `avg_annual_pay`
- `industry_code (NAICS supersector 10, 15, 20-90)`

**Mart columns fed by this source (9):**

- `emp_growth_yoy, emp_growth_yoy_demeaned`
- `wage_growth_yoy, wage_growth_yoy_demeaned`
- `high_wage_share, high_wage_share_yoy_chg`
- `knowledge_emp_yoy, knowledge_emp_cagr_3yr, knowledge_emp_growth_vs_state_pp`
- `renter_ind_emp_yoy, renter_ind_emp_cagr_3yr, renter_ind_emp_growth_vs_state_pp`
- `employment_diversity_hhi (Herfindahl index)`
- `n_supersectors_reporting, max_supersector_share`
- `anchor_type (gov/edu/healthcare/tourism), anchor_stability_score`
- `federal_gov_share, edu_share, healthcare_share, tourism_share`

**Notes:** Backbone of employment/wage/diversity features. Covers UI-covered employees (very complete).


#### Local Area Unemployment Statistics (LAUS) — *U.S. Bureau of Labor Statistics*

- **Staging table:** `stg_bls_laus`
- **Grain:** county × month (annualized)
- **Coverage:** ~1.4M rows
- **Intermediate models:** `int_bls_laus_unemployment`

**Fields extracted from source (1):**

- `unemployment_rate`

**Mart columns fed by this source (2):**

- `unemployment_rate`
- `unemployment_rate_yoy_chg`

**Notes:** Monthly county unemployment, aggregated to annual for mart.


#### Regional Income (CAINC1) — *U.S. Bureau of Economic Analysis*

- **Staging table:** `stg_bea_regional_income`
- **Grain:** county × annual
- **Coverage:** ~19K rows
- **Intermediate models:** `int_bea_income`

**Fields extracted from source (3):**

- `personal_income_thousands`
- `bea_population`
- `per_capita_income`

**Mart columns fed by this source (3):**

- `per_capita_income`
- `pcpi_yoy, pcpi_cagr_3yr, pcpi_cagr_5yr (⚠️ leaky)`
- `pcpi_growth_vs_state_1yr_pp, pcpi_cagr_3yr_vs_state_pp`

**Notes:** Primary per-capita income levels & growth. Own pop denominator (differs slightly from PEP).


#### Fair Market Rents (FMR) — *U.S. Department of Housing and Urban Development*

- **Staging table:** `stg_hud_fmr`
- **Grain:** county × annual (fiscal year)
- **Coverage:** ~23K rows
- **Intermediate models:** `int_hud_fmr` · `int_mf_features (mf_cap_rate_estimated)`

**Fields extracted from source (2):**

- `fmr_0br, fmr_1br, fmr_2br, fmr_3br, fmr_4br`
- `fmr_percentile (usually 40th)`

**Mart columns fed by this source (4):**

- `fmr_2br`
- `fmr_2br_yoy, fmr_2br_cagr_3yr, fmr_2br_cagr_3yr_vs_state_pp`
- `mf_cap_rate_estimated (FMR-based MF cap proxy)`
- `mf_cap_rate_change_3yr`

**Notes:** HUD 40th-%ile 2BR gross rent. Slow annual update. Backbone of MF-adjusted cap proxy.


#### House Price Index (HPI) — *Federal Housing Finance Agency (FHFA)*

- **Staging table:** `stg_fhfa_hpi`
- **Grain:** county × quarterly (annualized)
- **Coverage:** ~105K rows
- **Intermediate models:** `int_hpi_features`

**Fields extracted from source (2):**

- `hpi (all-transactions repeat-sales index)`
- `annual_change_pct`

**Mart columns fed by this source (6):**

- `hpi_yoy`
- `hpi_trailing_24mo`
- `hpi_5yr_cagr_deficit`
- `hpi_forward_return_24mo/36mo/60mo (targets)`
- `target_hpi_24/36/60mo_demeaned (ML targets)`
- `disq_price_ran_25pct (gate)`

**Notes:** Only source of long-term home price series. Repeat-sales, SFR-heavy but best available.


#### Zillow Observed Rent Index (ZORI) — *Zillow (private)*

- **Staging table:** `stg_zillow_zori`
- **Grain:** county × monthly (annualized)
- **Coverage:** ~93K rows. Coverage improves post-2015; sparse pre-2015
- **Intermediate models:** `int_zori_features`

**Fields extracted from source (1):**

- `zori (rent index, repeat-rent, MF-heavy inventory)`

**Mart columns fed by this source (4):**

- `zori_yoy`
- `zori_5yr_cagr_deficit`
- `zori_forward_return_24mo/36mo/60mo (targets)`
- `target_zori_24/36/60mo_demeaned (primary ML targets)`

**Notes:** Monthly, MF-adjacent. Complements ACS/FMR by being fresher but less deep.


#### IRS SOI Migration — *U.S. Internal Revenue Service*

- **Staging table:** `stg_irs_soi_migration`
- **Grain:** county × annual
- **Coverage:** ~12K rows
- **Intermediate models:** `int_irs_migration`

**Fields extracted from source (3):**

- `inflow_returns, inflow_exemptions, inflow_agi_thousands, inflow_agi_per_return`
- `outflow_returns, outflow_exemptions, outflow_agi_thousands, outflow_agi_per_return`
- `net_migration_returns, net_agi_thousands`

**Mart columns fed by this source (5):**

- `irs_net_migration_rate`
- `irs_inflow_rate`
- `irs_income_gap (inflow AGI − outflow AGI)`
- `irs_income_gap_pct`
- `irs_net_agi_per_capita`

**Notes:** Taxpayer-return migration flows with AGI — reveals income quality of movers.


#### National Risk Index (NRI) — *FEMA*

- **Staging table:** `stg_fema_nri`
- **Grain:** county (snapshot)
- **Coverage:** ~3,200 counties × 1 vintage
- **Intermediate models:** `(not yet integrated)`

**Fields extracted from source (4):**

- `risk_score, risk_ratng, risk_spctl (composite)`
- `eal_score, eal_valt (expected annual loss)`
- `resl_score, sovi_score (resilience, social vulnerability)`
- `18 hazards: avln, cfld, cwav, drgt, erqk, hail, hrcn, hwav, istm, lnds, ltng, swnd, trnd, tsun, vlcn, wfir, wntw`

**Mart columns fed by this source (1):**

- `(none currently — future work: climate/insurance risk overlay)`

**Notes:** Loaded but not yet integrated. Available for future climate-adjusted underwriting.


#### FAA Enplanements — *Federal Aviation Administration*

- **Staging table:** `stg_faa_enplanements`
- **Grain:** county × annual
- **Coverage:** ~2.9K rows
- **Intermediate models:** `int_enp_permits`

**Fields extracted from source (3):**

- `total_enplanements`
- `num_airports`
- `largest_hub_tier (Large/Medium/Small/Non-hub)`

**Mart columns fed by this source (1):**

- `enp_yoy, enp_cagr_3yr, enp_growth_vs_natl_pp`

**Notes:** Airport passenger traffic — proxy for connectivity/tourism/business travel. Only counties with commercial airports.


#### FRED Macro Series — *Federal Reserve Bank of St. Louis*

- **Staging table:** `stg_fred_macro`
- **Grain:** national × series × observation_date
- **Coverage:** ~30K rows across multiple national series
- **Intermediate models:** `int_national_macro_regime`

**Fields extracted from source (1):**

- `value (per series_id: Fed funds, 10yr yield, mortgage rates, CPI, etc.)`

**Mart columns fed by this source (1):**

- `macro_regime (categorical: rising/falling/neutral; joined at run-time in analog_search)`

**Notes:** National context. Tags each cohort year with a rate regime for tie-breaker analysis.


#### HUD USPS Vacancy — *U.S. Department of Housing and Urban Development*

- **Staging table:** `(not loaded — blocked)`
- **Grain:** ZIP × quarterly (would be)
- **Coverage:** None — requires HUD gov/nonprofit/edu-only USPS Vacancies registration
- **Intermediate models:** `int_zip_vacancy (skeleton exists)`

**Fields extracted from source (1):**

- `vacancy_rate (residential + business)`

**Mart columns fed by this source (1):**

- `vt_vacancy_level, vt_vacancy_change_3yr, vt_vacancy_change_5yr (currently null/sparse)`

**Notes:** ⚠️ Deferred — HUD registration barrier. Framework ready; awaiting access.


### 4.2 Live staging-table row counts (sanity check)

In [ ]:
staging_tables = [
    'stg_census_acs', 'stg_census_acs_1yr', 'stg_census_pep',
    'stg_census_bps', 'stg_census_bfs',
    'stg_bls_qcew', 'stg_bls_laus',
    'stg_bea_regional_income',
    'stg_hud_fmr', 'stg_fhfa_hpi', 'stg_zillow_zori',
    'stg_irs_soi_migration', 'stg_fema_nri',
    'stg_faa_enplanements', 'stg_fred_macro',
]
counts = []
for t in staging_tables:
    n = con.execute(f'SELECT COUNT(*) FROM main_staging.{t}').fetchone()[0]
    ncols = len(con.execute(f'DESCRIBE main_staging.{t}').df())
    counts.append({'staging_table': t, 'n_rows': f'{n:,}', 'n_cols': ncols})
pd.DataFrame(counts)

## 5. PIT-correctness notes

The intermediate models below emit **one row per (fips, scoring_period)** with as-of-aware source lookups (rewritten 2026-09-02). Safe to use as ML inputs:

- `int_acs_features`: rental_vacancy_rate, rent_to_income_ratio, cap_rate_proxy
- `int_mf_features`: renter_share, mf_permit_ratio, rental_vacancy_change_3yr, mf_cap_rate_estimated
- `int_mf_inventory`: MF building estimates + age

**⚠️ NOT rebuilt PIT-safe** (treat as leaky for training):
- `pcpi_cagr_5yr` — from `int_bea_income`, still latest-per-fips
- `n_supersectors_reporting` — from `int_employment_diversity`, likely latest-per-fips